# 🚀 Phase 2B: Track B Industrial High-Throughput Scalability ($N \ge 100\text{k}$)
## *Task-Technology Fit Analysis of Modern AI-Driven Intrusion Detection*
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

### 📌 Objectives:
- Stress-test scalable architectures: `Mambular SSM` ($O(L)$), `FT-Transformer` (mini-batch), `GraphIDS`, `XGBoost` (Hist), `LightGBM` (GPU Hist).
- Scale across sample dimensions $N \in \{25\text{k}, 50\text{k}, 100\text{k}\}$ flows.
- **Real vs. Synthetic Detection**: Stream directly from authentic decontaminated dataset (`data/processed/CICIDS2017_cleaned.parquet`) if present, otherwise fall back to synthetic partition with clear notices.
- Profile inference throughput ($\text{flows/sec}$), per-flow latency ($\text{ms}$), and peak GPU VRAM ($\text{MB}$).
- Output scalability curves and export LaTeX tables to Drive.


### 1. ☁️ Google Drive Mount & Project Root Auto-Resolution


In [ ]:
import os, sys
from pathlib import Path

# 1. Mount Google Drive if running in Colab
try:
    from google.colab import drive
    if not Path('/content/drive').exists() and not Path('/content/My Drive').exists():
        drive.mount('/content/drive')
except ImportError:
    print("ℹ️ Running in local/workstation environment.")

# 2. Candidate root paths (prioritizing user's Colab Notebooks directory)
CANDIDATE_ROOTS = [
    Path('/content/My Drive/Colab Notebooks'),
    Path('/content/drive/My Drive/Colab Notebooks'),
    Path('/content/drive/MyDrive/Colab Notebooks'),
    Path('/content/drive/MyDrive/is_ai-vuln'),
    Path('/content/is_ai-vuln'),
    Path('.').resolve()
]

PROJECT_ROOT = None
for cand in CANDIDATE_ROOTS:
    if cand.exists() and (cand / 'src').exists():
        PROJECT_ROOT = cand.resolve()
        break

if PROJECT_ROOT is None:
    PROJECT_ROOT = Path('.').resolve()

os.chdir(str(PROJECT_ROOT))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("=" * 80)
print(f"✅ Active Project Root : {PROJECT_ROOT}")
print(f"✅ src/ directory found: {(PROJECT_ROOT / 'src').exists()}")
print(f"📁 Data directory      : {(PROJECT_ROOT / 'data').resolve()}")
print("=" * 80)


### 2. 📦 Dependencies Installation


In [ ]:
!pip install -q xgboost lightgbm scikit-learn pandas numpy matplotlib seaborn
print("✅ Core dependencies installed.")


### 3. 🌊 Streaming Chunk Loader Initialization (Real vs. Synthetic Detection)

> [!IMPORTANT]
> **Data Notice**:
> - If `data/processed/CICIDS2017_cleaned.parquet` exists from Phase 1, Track B will stream **Real NetFlow Records**.
> - Otherwise, it will synthesize a 100,000-flow benchmark partition with a prominent notice banner.


In [ ]:
from src.data.streaming_loader import StreamingChunkLoader, resolve_track_b_dataset

# Resolve target streaming file: automatically prefers real data, falls back to synthetic
data_file, is_synthetic = resolve_track_b_dataset(
    base_dir=PROJECT_ROOT,
    dataset_name="CICIDS2017",
    target_samples=100000
)

loader = StreamingChunkLoader(data_file, chunk_size=25000)
print(f"🌊 Streaming Loader active for: {data_file.name}")
print(f"📊 Dataset Provenance: {'SYNTHETIC FALLBACK' if is_synthetic else 'REAL AUTHENTIC BENCHMARK'}")


### 4. ⚡ High-Throughput Benchmark Loop ($N \ge 100\text{k}$)


In [ ]:
import time
import numpy as np
import pandas as pd
from src.models import get_model
from src.utils.environment import flush_memory

sample_scales = [25000, 50000, 100000]
scalable_models = ["XGBoost", "LightGBM", "Mambular_SSM", "FT_Transformer", "GraphIDS"]

scalability_records = []

for n_records in sample_scales:
    print(f"\n{'='*50}\n📊 Benchmarking Scale: N = {n_records:,} flows\n{'='*50}")
    
    # Read streaming slice
    X_list, y_list = [], []
    for X_c, y_c in loader.iter_chunks(max_total_records=n_records):
        X_list.append(X_c)
        y_list.append(y_c)
    X_scale = np.vstack(X_list)
    y_scale = np.concatenate(y_list)
    
    # Split train/test (80/20)
    split_idx = int(0.8 * len(X_scale))
    X_tr, y_tr = X_scale[:split_idx], y_scale[:split_idx]
    X_va, y_va = X_scale[split_idx:], y_scale[split_idx:]
    
    for model_name in scalable_models:
        print(f"  ▶️ Training {model_name} on {len(X_tr):,} samples...")
        t0 = time.perf_counter()
        model = get_model(model_name)
        model.fit(X_tr, y_tr)
        train_time = time.perf_counter() - t0
        
        # Profile inference throughput
        profile = model.profile_inference(X_va, warmup_runs=2, repeat_runs=5)
        
        scalability_records.append({
            "Sample Scale": n_records,
            "Model": model_name,
            "Train Time (s)": round(train_time, 2),
            "Throughput (flows/s)": profile["throughput_flows_sec"],
            "Latency (ms/flow)": profile["latency_ms_per_flow"],
            "Peak VRAM (MB)": profile["vram_peak_mb"],
            "Is_Synthetic": is_synthetic
        })
        print(f"    ✅ {model_name}: {profile['throughput_flows_sec']:,.1f} flows/s | Latency: {profile['latency_ms_per_flow']} ms")
        model.cleanup()
        flush_memory()

df_scalability = pd.DataFrame(scalability_records)
display(df_scalability)


### 5. 📈 Scalability Curves Visualization & Table Export


In [ ]:
import matplotlib.pyplot as plt
from src.visualization import set_publication_style, save_publication_figure, export_benchmark_to_latex

set_publication_style(is_double_column=False)

fig, ax = plt.subplots(figsize=(8, 4), dpi=150)
for m in scalable_models:
    sub = df_scalability[df_scalability["Model"] == m]
    ax.plot(sub["Sample Scale"], sub["Throughput (flows/s)"], marker="o", label=m)

ax.set_title(f"Track B: Inference Throughput Scaling (Data: {'SYNTHETIC' if is_synthetic else 'REAL'})")
ax.set_xlabel("Sample Dimension (N Records)")
ax.set_ylabel("Throughput (flows / second) [Higher is Better]")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)

out_dir = PROJECT_ROOT / "experiment_output" / "track_b_scalability"
out_dir.mkdir(parents=True, exist_ok=True)
save_publication_figure(fig, str(out_dir / "figure_throughput_scaling"))
export_benchmark_to_latex(df_scalability, out_dir / "scalability_table.tex", caption="Track B Throughput Scalability Profiling")
plt.show()
